In [1]:
import sys
print(sys.executable)

d:\Fraud-Triage\fraud-triage\.venv\Scripts\python.exe


In [5]:
from sklearn.metrics import average_precision_score,roc_auc_score
from fraud_triage.data.load import load_and_merge_raw_data
from fraud_triage.data.split import time_based_split
import lightgbm as lgb

merged_df = load_and_merge_raw_data()
train_df, val_df = time_based_split(merged_df)

drop_cols=["isFraud","TransactionID"]

x_train = train_df.drop(columns=drop_cols)
x_test = val_df.drop(columns=drop_cols)
y_train=train_df["isFraud"]

y_test=val_df["isFraud"]

cat_cols = x_train.select_dtypes(include="category").columns.tolist()

model = lgb.LGBMClassifier(random_state=42)
model.fit(x_train, y_train, categorical_feature=cat_cols)

valid_pred = model.predict_proba(x_test)[:, 1]
pr_auc = average_precision_score(y_test, valid_pred)
roc_auc = roc_auc_score(y_test, valid_pred)

import pandas as pd

results = pd.DataFrame({"y_true": y_test.values, "score": valid_pred})
results = results.sort_values("score", ascending=False)

top_k_pct = 0.01
top_k_n = int(len(results) * top_k_pct)
top_k = results.head(top_k_n)

precision_at_k = top_k["y_true"].sum() / top_k_n
recall_at_k = top_k["y_true"].sum() / results["y_true"].sum()

print(f"Precision@1%: {precision_at_k:.4f}")
print(f"Recall@1%:    {recall_at_k:.4f}")

D:\Fraud-Triage\fraud-triage\src\fraud_triage\data\load.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_df["day"] = merged_df["TransactionDT"] // (24 * 60 * 60)


Saved to: D:\Fraud-Triage\fraud-triage\Data\interim\train_merged.parquet
Shape: (590540, 435)
File size: 80.17 MB
Train: (440018, 435) Fraud rate: 0.03500993141189679
Val:   (142545, 435) Fraud rate: 0.03457153881230489
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 15405, number of negative: 424613
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.291282 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 35066
[LightGBM] [Info] Number of data points in the train set: 440018, number of used features: 431
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035010 -> initscore=-3.316486
[LightGBM] [Info] Start training from sco

In [ ]:
print(f"PR-AUC:  {pr_auc:.4f}")
print(f"ROC-AUC: {roc_auc:.4f}")


PR-AUC:  0.4963
ROC-AUC: 0.8978


: 